# DETR3D in Pure PyTorch: A Literate Architecture

This output-free notebook builds the faithful DETR3D architecture locally, one tensor contract at a time. Architecture code does not import the production model; its immutable reference is production commit `5244860688cfc390a82f8c730017853188b50e5f`. Reduced synthetic probes are the default, while real data, checkpoint, training, and official evaluation paths are explicit opt-ins.

The faithful image contract is important: camera images are resized to `900x1600`, then padded only along height to `928x1600` before entering the model. Projection geometry remains tied to the unpadded `900x1600` image.

```text
six resized images [B,6,3,900,1600]
  -> official padding [B,6,3,928,1600]
  |
  +-> GridMask during training
  |
  +-> shared Caffe-style ResNet-101 + modulated DCNv2
  |     C3 [B,6,512,116,200]
  |     C4 [B,6,1024,58,100]
  |     C5 [B,6,2048,29,50]
  |
  +-> FPN
  |     P3 [B,6,256,116,200]
  |     P4 [B,6,256,58,100]
  |     P5 [B,6,256,29,50]
  |     P6 [B,6,256,15,25]
  |
900 query/content + query-position embeddings [B,900,256]
  -> initial normalized 3D references [B,900,3]
  -> decoder layer x6
       self-attention
       project references into N cameras
       sample and fuse L pyramid levels
       feed-forward network
       class [B,900,10], encoded box [B,900,10]
       detach refined references for the next layer
  -> all class logits [6,B,900,10]
  -> all encoded boxes [6,B,900,10]
```

## Symbols And Execution Modes

| Symbol | Meaning | Faithful value |
|---|---|---:|
| `B` | batch size | training dependent |
| `N` | synchronized camera views | 6 |
| `Q` | object queries | 900 |
| `C` | decoder/FPN channels | 256 |
| `L` | FPN levels | 4 |
| `K` | nuScenes classes | 10 |
| `D` | encoded box coordinates | 10 |

`SMOKE_MODE` keeps probes deterministic and small. Every expensive or asset-dependent path has a separate switch and is disabled unless requested.

In [ ]:
from __future__ import annotations

import random
from typing import Dict, Iterable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models
from torchvision.models import ResNet101_Weights
from torchvision.ops import DeformConv2d

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SMOKE_MODE = True
RUN_REAL_DATA = False
RUN_CHECKPOINT = False
RUN_TRAIN_STEP = True
RUN_OFFICIAL_EVAL = False
NUSCENES_DATAROOT = None
CHECKPOINT_PATH = None
FCOS3D_CHECKPOINT_PATH = None

ARCH-GRID-MASK

## GridMask: Shared Structured Occlusion

GridMask draws one periodic binary canvas and broadcasts it across every flattened camera image and color channel. For grid spacing `d`, ratio `r`, and random phase `(s_h, s_w)`, the removed stripe width is

```text
l = clip(round(d * r), 1, d - 1)
horizontal stripe i: [i*d + s_h, i*d + s_h + l)
vertical stripe   j: [j*d + s_w, j*d + s_w + l)
```

A `1.5H x 1.5W` canvas permits rotation before a centered `H x W` crop. Production flattens cameras first, `[B,N,3,H,W] -> [B*N,3,H,W]`; the mask `[1,1,H,W]` therefore broadcasts over `B*N` and RGB. Applying this module directly to a five-dimensional probe has the equivalent broadcast `[1,1,1,H,W]`, so all views still share the sampled pattern. In `mode=1`, the mask is inverted. Evaluation mode and the probability gate return the original tensor unchanged.

In [ ]:
class GridMask(nn.Module):
    def __init__(
        self,
        use_h: bool = True,
        use_w: bool = True,
        rotate: int = 1,
        offset: bool = False,
        ratio: float = 0.5,
        mode: int = 1,
        probability: float = 0.7,
    ):
        super().__init__()
        self.use_h = use_h
        self.use_w = use_w
        self.rotate = rotate
        self.offset = offset
        self.ratio = ratio
        self.mode = mode
        self.probability = probability

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if not self.training or np.random.rand() > self.probability:
            return images
        height, width = images.shape[-2:]
        canvas_height = int(1.5 * height)
        canvas_width = int(1.5 * width)
        spacing = np.random.randint(2, height)
        line_width = min(max(int(spacing * self.ratio + 0.5), 1), spacing - 1)
        mask = np.ones((canvas_height, canvas_width), dtype=np.float32)
        start_h = np.random.randint(spacing)
        start_w = np.random.randint(spacing)
        if self.use_h:
            for index in range(canvas_height // spacing):
                start = spacing * index + start_h
                mask[start : min(start + line_width, canvas_height), :] = 0
        if self.use_w:
            for index in range(canvas_width // spacing):
                start = spacing * index + start_w
                mask[:, start : min(start + line_width, canvas_width)] = 0

        rotation = np.random.randint(self.rotate) if self.rotate > 1 else 0
        mask = np.asarray(Image.fromarray(mask).rotate(rotation)).copy()
        top = (canvas_height - height) // 2
        left = (canvas_width - width) // 2
        mask = mask[top : top + height, left : left + width]
        mask_tensor = torch.as_tensor(mask, device=images.device, dtype=images.dtype)
        if self.mode == 1:
            mask_tensor = 1 - mask_tensor
        mask_tensor = mask_tensor.view(1, 1, height, width)  # [H,W] -> [1,1,H,W]; broadcasts over flattened images and RGB
        if not self.offset:
            return images * mask_tensor  # [B*N,3,H,W] x [1,1,H,W] -> [B*N,3,H,W]

        noise = torch.as_tensor(
            2 * (np.random.rand(height, width) - 0.5),
            device=images.device,
            dtype=images.dtype,
        ).view(1, 1, height, width)  # [H,W] -> [1,1,H,W]; shared additive noise
        return images * mask_tensor + noise * (1 - mask_tensor)  # [B*N,3,H,W]


In [ ]:
grid_probe = GridMask(probability=1.0)
grid_images = torch.ones(2, 3, 3, 16, 24)
grid_probe.eval()
grid_eval = grid_probe(grid_images)
assert grid_eval is grid_images
grid_probe.train()
grid_train = grid_probe(grid_images)
assert grid_train.shape == (2, 3, 3, 16, 24)
assert torch.equal(grid_train[:, 0], grid_train[:, 1])

ARCH-BACKBONE

## Caffe ResNet-101 And Modulated DCNv2

Each ResNet bottleneck normally samples a fixed `k x k` grid. DCNv2 predicts an `(x,y)` displacement and a sigmoid modulation value for each of the `k^2` locations:

```text
conv_offset(x) = concat(offset, mask_logits) [B*N,3*k^2,H_o,W_o]
offset                                  [B*N,2*k^2,H_o,W_o]
mask = sigmoid(mask_logits)             [B*N,  k^2,H_o,W_o]
deform_conv(x, offset, mask)            [B*N,C_o,H_o,W_o]
```

Zero-initialized offset/mask prediction starts with zero displacement and modulation `sigmoid(0)=0.5`, matching the pinned production module. The original dense convolution weights are copied into `deform_conv`. The offset-only `DeformConv2dPack` is retained because it is the production fallback when `official_style=False`; the faithful path selects the modulated pack.

Official Caffe stride placement moves each downsampling stride from bottleneck `conv2` to `conv1`. Stage 4 and stage 5 then replace every `conv2` with modulated DCNv2. Cameras share one backbone by flattening `B` and `N`, then restoring the camera axis:

```text
images [B,N,3,H,W] -> [B*N,3,H,W] -> shared ResNet -> restore B,N
faithful input [B,6,3,928,1600]
  C3 [B,6, 512,116,200]
  C4 [B,6,1024, 58,100]
  C5 [B,6,2048, 29, 50]
```

The stem is frozen for `frozen_stages >= 0`. Calling `train()` re-applies freezing, and `norm_eval=True` keeps every BatchNorm layer in evaluation mode so running statistics do not drift.

In [ ]:
class DeformConv2dPack(nn.Module):
    """Offset-only deformable convolution used by the non-official fallback."""

    def __init__(self, conv: nn.Conv2d):
        super().__init__()
        kernel_h, kernel_w = conv.kernel_size
        offset_channels = 2 * kernel_h * kernel_w
        self.offset_conv = nn.Conv2d(
            in_channels=conv.in_channels,
            out_channels=offset_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            groups=1,
            bias=True,
        )
        self.deform_conv = DeformConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=kernel_h,
            stride=conv.stride[0],
            padding=conv.padding[0],
            dilation=conv.dilation[0],
            groups=conv.groups,
            bias=conv.bias is not None,
        )
        with torch.no_grad():
            self.deform_conv.weight.copy_(conv.weight)
            if conv.bias is not None and self.deform_conv.bias is not None:
                self.deform_conv.bias.copy_(conv.bias)
            nn.init.zeros_(self.offset_conv.weight)
            nn.init.zeros_(self.offset_conv.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        offset = self.offset_conv(x)  # [B*N,C_i,H,W] -> [B*N,2*k^2,H_o,W_o]
        return self.deform_conv(x, offset)  # [B*N,C_i,H,W] -> [B*N,C_o,H_o,W_o]


class ModulatedDeformConv2dPack(nn.Module):
    def __init__(self, conv: nn.Conv2d):
        super().__init__()
        kernel_h, kernel_w = conv.kernel_size
        self.offset_channels = 2 * kernel_h * kernel_w
        self.mask_channels = kernel_h * kernel_w
        self.conv_offset = nn.Conv2d(
            in_channels=conv.in_channels,
            out_channels=self.offset_channels + self.mask_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            groups=1,
            bias=True,
        )
        self.deform_conv = DeformConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            groups=conv.groups,
            bias=conv.bias is not None,
        )
        with torch.no_grad():
            self.deform_conv.weight.copy_(conv.weight)
            if conv.bias is not None and self.deform_conv.bias is not None:
                self.deform_conv.bias.copy_(conv.bias)
            nn.init.zeros_(self.conv_offset.weight)
            nn.init.zeros_(self.conv_offset.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        offset_mask = self.conv_offset(x)  # [B*N,C_i,H,W] -> [B*N,3*k^2,H_o,W_o]
        offset = offset_mask[:, : self.offset_channels]  # [B*N,3*k^2,H_o,W_o] -> [B*N,2*k^2,H_o,W_o]
        mask = offset_mask[:, self.offset_channels :].sigmoid()  # [B*N,3*k^2,H_o,W_o] -> [B*N,k^2,H_o,W_o]
        return self.deform_conv(x, offset, mask)  # [B*N,C_i,H,W] -> [B*N,C_o,H_o,W_o]


In [ ]:
class MultiViewImageBackbone(nn.Module):
    """Apply one torchvision ResNet-101 to every camera view."""

    def __init__(
        self,
        variant: str = "resnet101",
        pretrained: bool = True,
        frozen_stages: int = 1,
        norm_eval: bool = True,
        official_style: bool = False,
    ):
        super().__init__()
        if variant != "resnet101":
            raise ValueError(f"Unsupported backbone variant: {variant}")

        weights = ResNet101_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = models.resnet101(weights=weights)
        if official_style:
            self._use_caffe_stride(backbone)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.stage2 = backbone.layer1
        self.stage3 = backbone.layer2
        self.stage4 = backbone.layer3
        self.stage5 = backbone.layer4
        self._convert_stage_to_deformable(self.stage4, modulated=official_style)
        self._convert_stage_to_deformable(self.stage5, modulated=official_style)
        self.frozen_stages = frozen_stages
        self.norm_eval = norm_eval
        self._freeze_stages()

    @staticmethod
    def _use_caffe_stride(backbone: nn.Module) -> None:
        for stage in (backbone.layer2, backbone.layer3, backbone.layer4):
            block = stage[0]
            block.conv1.stride = (2, 2)
            block.conv2.stride = (1, 1)

    @staticmethod
    def _convert_stage_to_deformable(stage: nn.Sequential, *, modulated: bool) -> None:
        pack_type = ModulatedDeformConv2dPack if modulated else DeformConv2dPack
        for block in stage:
            if hasattr(block, "conv2") and isinstance(block.conv2, nn.Conv2d):
                block.conv2 = pack_type(block.conv2)

    def _freeze_stages(self) -> None:
        if self.frozen_stages >= 0:
            self.stem.eval()
            for param in self.stem.parameters():
                param.requires_grad = False
        stages = [self.stage2, self.stage3, self.stage4, self.stage5]
        for idx, stage in enumerate(stages, start=1):
            if self.frozen_stages >= idx:
                stage.eval()
                for param in stage.parameters():
                    param.requires_grad = False

    def train(self, mode: bool = True):
        super().train(mode)
        self._freeze_stages()
        if mode and self.norm_eval:
            for module in self.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images: torch.Tensor) -> Dict[str, torch.Tensor]:
        batch, num_cams, channels, height, width = images.shape
        flat = images.reshape(batch * num_cams, channels, height, width)  # [B,N,3,H,W] -> [B*N,3,H,W]; faithful [B,6,3,928,1600]

        x = self.stem(flat)  # [B*N,3,H,W] -> [B*N,64,H/4,W/4]
        stage2 = self.stage2(x)  # [B*N,64,H/4,W/4] -> [B*N,256,H/4,W/4]
        stage3 = self.stage3(stage2)  # [B*N,256,H/4,W/4] -> [B*N,512,H/8,W/8]
        stage4 = self.stage4(stage3)  # [B*N,512,H/8,W/8] -> [B*N,1024,H/16,W/16]
        stage5 = self.stage5(stage4)  # [B*N,1024,H/16,W/16] -> [B*N,2048,H/32,W/32]

        features = {"stage3": stage3, "stage4": stage4, "stage5": stage5}
        multi_view = {}
        for name, feat in features.items():
            _, out_channels, out_height, out_width = feat.shape
            multi_view[name] = feat.reshape(batch, num_cams, out_channels, out_height, out_width)  # [B*N,C_l,H_l,W_l] -> [B,N,C_l,H_l,W_l]
        return multi_view


ARCH-FPN

## ImageFPN: A Four-Level Top-Down Pyramid

The feature pyramid first projects all backbone stages to `C=256`, then carries coarse semantic information toward finer maps:

```text
L_l = Conv1x1(C_l)
T_l = L_l + nearest_upsample(T_(l+1))
P_l = Conv3x3(T_l)
P6  = Conv3x3_stride2(ReLU(P5))

C5 ---- 1x1 ---- T5 ---- 3x3 ---- P5 ---- ReLU ---- stride-2 ---- P6
                   | upsample
C4 ---- 1x1 ---- (+) ---- 3x3 ---- P4
                   | upsample
C3 ---- 1x1 ---- (+) ---- 3x3 ---- P3
```

The pinned class exposes `relu_before_extra_convs`; faithful construction sets it to `True`. For padded `928x1600` inputs the restored multi-view outputs are `P3 [B,6,256,116,200]`, `P4 [B,6,256,58,100]`, `P5 [B,6,256,29,50]`, and `P6 [B,6,256,15,25]`.

In [ ]:
class ImageFPN(nn.Module):
    def __init__(
        self,
        in_channels: Iterable[int] = (512, 1024, 2048),
        out_channels: int = 256,
        out_names: Iterable[str] = ("p3", "p4", "p5", "p6"),
        relu_before_extra_convs: bool = False,
    ):
        super().__init__()
        self.out_names = list(out_names)
        self.relu_before_extra_convs = relu_before_extra_convs
        self.lateral_convs = nn.ModuleList(
            [nn.Conv2d(ch, out_channels, kernel_size=1) for ch in in_channels]
        )
        self.output_convs = nn.ModuleList(
            [nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1) for _ in in_channels]
        )
        self.extra_conv = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=2, padding=1
        )

    def forward(self, features: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        stage_names = sorted(features.keys())
        flat_feats = []
        batch, num_cams = next(iter(features.values())).shape[:2]
        for name in stage_names:
            feat = features[name]
            flat = feat.reshape(batch * num_cams, feat.shape[2], feat.shape[3], feat.shape[4])  # [B,N,C_l,H_l,W_l] -> [B*N,C_l,H_l,W_l]
            flat_feats.append(flat)

        laterals = []
        for conv, feat in zip(self.lateral_convs, flat_feats):
            lateral = conv(feat)  # [B*N,C_l,H_l,W_l] -> [B*N,256,H_l,W_l]
            laterals.append(lateral)
        for idx in range(len(laterals) - 1, 0, -1):
            upsampled = F.interpolate(laterals[idx], size=laterals[idx - 1].shape[-2:], mode="nearest")  # [B*N,256,H_(l+1),W_(l+1)] -> [B*N,256,H_l,W_l]
            laterals[idx - 1] = laterals[idx - 1] + upsampled  # [B*N,256,H_l,W_l]

        outputs = []
        for conv, feat in zip(self.output_convs, laterals):
            output = conv(feat)  # [B*N,256,H_l,W_l] -> [B*N,256,H_l,W_l]
            outputs.append(output)
        extra_input = F.relu(outputs[-1]) if self.relu_before_extra_convs else outputs[-1]
        outputs.append(self.extra_conv(extra_input))  # [B*N,256,H_5,W_5] -> [B*N,256,ceil(H_5/2),ceil(W_5/2)]

        pyramid: Dict[str, torch.Tensor] = {}
        for out_name, feat in zip(self.out_names, outputs):
            pyramid[out_name] = feat.reshape(batch, num_cams, feat.shape[1], feat.shape[2], feat.shape[3])  # [B*N,256,H_l,W_l] -> [B,N,256,H_l,W_l]
        return pyramid


## Reduced Deterministic Image-Path Probe

A single `64x96` camera image preserves the real ResNet-101/DCNv2/FPN channel contracts while reducing spatial work. Evaluation mode disables stochastic and BatchNorm updates. The expected backbone sizes are `8x12`, `4x6`, and `2x3`; stride-2 P6 is therefore `1x2`.

In [ ]:
if SMOKE_MODE:
    smoke_backbone = MultiViewImageBackbone(
        pretrained=False, official_style=True
    ).eval()
    smoke_fpn = ImageFPN(relu_before_extra_convs=True).eval()
    smoke_images = torch.randn(1, 1, 3, 64, 96)
    with torch.inference_mode():
        smoke_features = smoke_backbone(smoke_images)
        smoke_pyramid = smoke_fpn(smoke_features)

    assert list(smoke_features) == ["stage3", "stage4", "stage5"]
    assert [tensor.ndim for tensor in smoke_features.values()] == [5, 5, 5]
    assert [tensor.shape[:2] for tensor in smoke_features.values()] == [(1, 1)] * 3
    assert [tensor.shape[-2:] for tensor in smoke_features.values()] == [(8, 12), (4, 6), (2, 3)]
    assert list(smoke_pyramid) == ["p3", "p4", "p5", "p6"]
    assert all(tensor.ndim == 5 for tensor in smoke_pyramid.values())
    assert all(tensor.shape[:3] == (1, 1, 256) for tensor in smoke_pyramid.values())
    pyramid_sizes = [tensor.shape[-2:] for tensor in smoke_pyramid.values()]
    assert pyramid_sizes == [(8, 12), (4, 6), (2, 3), (1, 2)]
    assert all(
        fine_h > coarse_h and fine_w > coarse_w
        for (fine_h, fine_w), (coarse_h, coarse_w) in zip(
            pyramid_sizes, pyramid_sizes[1:]
        )
    )
    assert smoke_pyramid["p6"].shape == (1, 1, 256, 1, 2)